In [4]:
import arcpy
import re
import os
import pandas as pd

In [5]:
arcpy.env.workspace = r"D:\USLULC"

In [6]:
raster_list = arcpy.ListRasters("*", "TIF")

In [7]:
for raster in raster_list:
    full_path = os.path.join(arcpy.env.workspace, raster)
    print(full_path)

D:\USLULC\Annual_NLCD_LndCov_1985_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1986_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1987_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1988_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1989_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1990_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1991_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1992_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1993_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1994_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1995_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1996_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1997_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1998_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_1999_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_2000_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_2001_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_2002_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_2003_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_2004_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_2005_CU_C1V0.tif
D:\USLULC\Annual_NLCD_LndCov_2006_

In [ ]:
# --- 1. Workspace and Environment Setup ---
# It's good practice to allow overwriting of outputs during testing
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
# --- 2. Define your inputs and outputs ---
# Replace these paths with the actual paths to your data
county_fc = r"C:\Users\jbiswas\UNC Charlotte Dropbox\Jayanta Biswas\GeoVis\GeoVis\GeoVis.gdb\USACensusTractNC"      # Your US States polygon layer
output_folder = r"C:\Users\jbiswas\Downloads\Charlotte_LC"  # Where the output rasters will go

os.makedirs(output_folder, exist_ok=True)

# The exact name of the attribute field containing the State Name or Abbreviation
county_name_field = "TRACT_FIPS" 

# --- 3. Create a Feature Layer ---
# We must create a feature layer because you cannot make selections directly on a feature class on disk
arcpy.management.MakeFeatureLayer(county_fc, "county_lyr")

# --- 4. Loop through the states ---
# Use a SearchCursor to read the state names from the attribute table
print("Starting the clipping process...")

for input_raster in raster_list:
    filename = os.path.basename(input_raster)

    # Search for exactly 4 digits (\d{4})
    match = re.search(r'\d{4}', filename)
    
    if match:
        year = match.group(0)
        print(year)
    with arcpy.da.SearchCursor("county_lyr", [county_name_field]) as cursor:
        for row in cursor:
            county_name = row[0]
            
            # Clean the state name to ensure it's a valid Windows filename 
            # (e.g., removes spaces so "New York" becomes "NewYork")
            safe_name = "".join(c for c in county_name if c.isalnum())
            
            print(f"Processing: {county_name}...")
            
            # --- 5. Select the current state ---
            # Construct the SQL query to select the specific state
            where_clause = f'"{county_name_field}" = \'{county_name}\''
            arcpy.management.SelectLayerByAttribute("county_lyr", "NEW_SELECTION", where_clause)
            
            # --- 6. Define the output path ---
            out_raster_path = os.path.join(output_folder, f"clipped_{safe_name}_{year}.tif")
            
            # --- 7. Clip the Raster ---
            try:
                arcpy.management.Clip(
                    in_raster=input_raster,
                    rectangle="#",                     # '#' tells it to use the extent of the selected state
                    out_raster=out_raster_path,
                    in_template_dataset="county_lyr",  # Uses the currently selected state in the layer
                    nodata_value="#",
                    clipping_geometry="ClippingGeometry", # CRITICAL: Clips to the polygon boundary, not just the bounding box
                    maintain_clipping_extent="MAINTAIN_EXTENT"
                )
                print(f"  -> Successfully created: {out_raster_path}")
                
            except arcpy.ExecuteError:
                print(f"  -> FAILED to clip {county_name}.")
                print(arcpy.GetMessages(2)) # Prints the specific ArcPy error
    
    # --- 8. Cleanup ---
    # Clear the selection and delete the temporary layer from memory
    arcpy.management.SelectLayerByAttribute("county_lyr", "CLEAR_SELECTION")
arcpy.management.Delete("county_lyr")
    
print("All processes complete!")

Starting the clipping process...
1985
Processing: 005609...
  -> Successfully created: C:\Users\jbiswas\Downloads\Charlotte_LC\clipped_005609_1985.tif
Processing: 001603...
  -> Successfully created: C:\Users\jbiswas\Downloads\Charlotte_LC\clipped_001603_1985.tif
Processing: 005618...
  -> Successfully created: C:\Users\jbiswas\Downloads\Charlotte_LC\clipped_005618_1985.tif
Processing: 003013...
  -> Successfully created: C:\Users\jbiswas\Downloads\Charlotte_LC\clipped_003013_1985.tif
Processing: 005510...
  -> Successfully created: C:\Users\jbiswas\Downloads\Charlotte_LC\clipped_005510_1985.tif
Processing: 003015...
  -> Successfully created: C:\Users\jbiswas\Downloads\Charlotte_LC\clipped_003015_1985.tif
Processing: 005613...
  -> Successfully created: C:\Users\jbiswas\Downloads\Charlotte_LC\clipped_005613_1985.tif
Processing: 002909...
  -> Successfully created: C:\Users\jbiswas\Downloads\Charlotte_LC\clipped_002909_1985.tif
Processing: 003018...
  -> Successfully created: C:\Users\